In [ ]:
!pip install langchain-graphrag==0.0.9 graspologic networkx langchain-ollama langchain-community langchain

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
spath = '/content/drive/My Drive/genaiproj/save'

Mounted at /content/drive


In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,442 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,935 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
# !ollama pull llama3:8b
# !ollama pull llama3.1:8b
!ollama pull mistral:7b
!ollama list


NAME          ID              SIZE      MODIFIED               
mistral:7b    6577803aa9a0    4.4 GB    Less than a second ago    


In [ ]:
import os
# Path to the problematic file within langchain_graphrag
file_to_patch = '/usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py'

# Check if the file exists
if os.path.exists(file_to_patch):
    print(f"Patching: {file_to_patch}")
    with open(file_to_patch, 'r') as f:
        content = f.read()

    # Replace the old import with the new one
    # Only patch if the old import line is present and the new one is not already there
    old_import = 'from langchain.output_parsers import PydanticOutputParser'
    new_import = 'from langchain_core.output_parsers import PydanticOutputParser'

    if old_import in content and new_import not in content:
        content = content.replace(old_import, new_import)
        with open(file_to_patch, 'w') as f:
            f.write(content)
        print("Successfully patched langchain_graphrag for output_parsers compatibility.")
    elif new_import in content:
        print("Patch already applied.")
    else:
        print("Old import line not found, no patch applied (or file structure changed unexpectedly).")
else:
    print(f"File not found: {file_to_patch}. Manual patching may be required or the environment is different.")

Patching: /usr/local/lib/python3.12/dist-packages/langchain_graphrag/indexing/report_generation/_output_parser.py
Successfully patched langchain_graphrag for output_parsers compatibility.


In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.cache import SQLiteCache

PATH_FOLDER = '/content/drive/MyDrive/genaiproj/save'
llm_model = "mistral:7b"  # You can change to gpt-4o, gpt-3.5-turbo, etc.
print(f"Initializing model {llm_model}...")
llm = ChatOllama(model=llm_model, temperature=0, seed=42)


Initializing model mistral:7b...


In [ ]:
import os
import pickle
import networkx as nx

master_path = os.path.join(spath, 'master_graph_all.pkl')

if os.path.exists(master_path):
    with open(master_path, 'rb') as f:
        final_graph = pickle.load(f)
    print(f"✅ Success! Loaded Master Graph with {final_graph.number_of_nodes()} nodes.")
else:
    print("❌ Error: File not found. Check your Drive path.")

✅ Success! Loaded Master Graph with 4593 nodes.


In [ ]:
# Look at one random node's data
sample_node = list(final_graph.nodes(data=True))[0]
print(f"Sample Node: {sample_node}")

In [ ]:
import pandas as pd
import os

spath = '/content/drive/MyDrive/genaiproj/save'
load_path = os.path.join(spath, 'df_communities_summarized_all.pkl')

# Instantly load the summarized dataframe
df_communities = pd.read_pickle(load_path)
print(f"✅ Loaded {len(df_communities)} summarized communities!")

✅ Loaded 1406 summarized communities!


#test for local

In [ ]:
    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    STOP_WORDS = {"is", "of", "to", "in", "it", "on", "as", "at", "be", "do", "an", "or", "by", "if", "a", "the", "and", "for", "with", "are", "how", "what", "does"}

    matched_nodes = []
    for node in final_graph.nodes():
        node_str = str(node).lower()

        # Break the node into words
        node_words = set(re.findall(r'\b\w+\b', node_str))

        # Remove the filler words from the node's words
        meaningful_node_words = node_words - STOP_WORDS

        if node_str in query.lower() or any(word in query_words for word in meaningful_node_words):
            matched_nodes.append(node)

# Local Search

In [ ]:
def local_search(query: str, final_graph, llm, top_k=5):
    print(f"Executing Local Search for: '{query}'")

    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    # 1. Safe Entity Matching (No external functions needed)
    matched_nodes = []
    for node in final_graph.nodes():
        if str(node).lower() in query.lower():
            matched_nodes.append(node)

    print(f"Detected query entities: {matched_nodes}")

    if not matched_nodes:
        return "I couldn't find any specific entities in your query to search the graph for. Try rephrasing with specific terms from the paper."

    # 2. Extract the local neighborhood (1-hop relationships)
    context_relationships = []
    for node in matched_nodes[:top_k]:
        all_edges = list(final_graph.edges(node, data=True))
        for u, v, data in all_edges[:15]:
            context_relationships.append(f"{u} --[{data.get('label', 'related_to')}]-- {v}")

    # 3. Construct the prompt
    context_str = "\n".join(set(context_relationships))
    prompt = f"""
    You are an experienced, highly adaptable clinical specialist.

    CRITICAL INSTRUCTIONS FOR YOUR TONE AND STYLE:
    1. MIRROR THE USER'S EXPERTISE: You must analyze the vocabulary and complexity of the Question.
       - If the question uses simple, everyday language or informal terms (e.g., sugar issues, pee a lot, stressed), respond like a friendly, empathetic diabetes educator using simple, conversational language. Translate all medical jargon.
       - If the question uses formal clinical terminology (e.g., glycemic status, euglycemic DKA, etiology), respond like a specialist speaking to a colleague using precise, professional medical terminology.
    2. Be concise and direct. Do not use generic academic filler (e.g., comprehensive assessment, multifaceted approach).
    3. Write naturally. Avoid overly robotic formatting and strict bulleted lists; use smooth paragraph transitions instead.
    4. Never start your response with "Based on the context... or According to the provided text...."
    5. ABSOLUTE FORMATTING BAN: You are STRICTLY PROHIBITED from using bullet points (-), numbered lists (1., 2.), or bold text headers. You MUST write in continuous, flowing paragraphs. Failure to do so is a critical error.

    EXAMPLE OF THE PERFECT FORMAT:
    User: How do I check my blood sugar?
    Response: To check how your blood sugar is doing, your doctor will likely run an A1C test, which gives a great average of your levels over the last few months. For day-to-day tracking, you can use a regular glucose monitor with a simple finger-prick, or you might use a continuous monitor that you wear on your arm to get real-time updates.


    CONTEXT:
    {context_str}

    QUESTION: {query}

    Answer strictly using the context.
    """

    prompt_template = f"""
    You are a helpful medical assistant. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. Do not make up information.


    Context:
    {context_str}


    Question:
    {query}


    Answer:
    """

    edges = context_relationships

    format_edges = "Retrieved Relationships:\n" + "\n".join(edges)

    print("Querying Ollama...")
    response = llm.invoke(prompt_template)
    return response.content, [format_edges]

# Test it!
print("-----")
test_query = "Does diabetes associate with cataract?"
print(local_search(test_query, final_graph, llm))

-----
Executing Local Search for: 'Does diabetes associate with cataract?'
Detected query entities: ['DIABETES CARE', '2026 DIABETES CARE', 'DIABETES', 'PEOPLE WITH DIABETES', 'AMERICAN DIABETES ASSOCIATION (ADA)', 'PROFESSIONAL DIABETES ORGANIZATION', 'AMERICAN DIABETES ASSOCIATION', 'TAR', 'ACTION IN DIABETES AND VASCULAR DISEASE: MODIFYING FACTORS FAVOR MORE STRINGENT GOAL', 'SHORT DIABETES DURATION', 'LONG DIABETES DURATION', 'DIABETES SELF-MANAGEMENT', 'INDIVIDUALS WITH DIABETES', 'PATIENT WITH DIABETES', 'TYPE 1 DIABETES', 'DIABETES SELF-MANAGEMENT EDUCATION AND SUPPORT PROGRAM', 'TRAINED DIABETES CARE AND EDUCATION SPECIALIST', 'DIABETES TREATMENT', 'DIABETES CARE AND EDUCATION SPECIALISTS', 'TYPE 2 DIABETES', 'PREGESTATIONAL DIABETES', 'GESTATIONAL DIABETES', 'DIABETES CARE TEAM', 'STANDARDS OF CARE IN DIABETES', 'DIABETES JOURNALS ORG', 'PROFESSIONAL PRACTICE COMMITTEE FOR DIABETES', 'DIABETES MELLITUS', 'TYP 2 DIABETES', 'SCREENING AND DIAGNOSIS OF DIABETES', 'DIABETES CONTRO

# New Global search

In [ ]:
import re

def global_search(query: str, df_comms, llm, top_n=3):
    print(f"Executing Global Search for: '{query}'")

    # 1. Clean and tokenize the user's query into a set of lowercase keywords
    # (Ignoring common tiny words like 'the', 'is', 'a')
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    stop_words = {'what', 'are', 'the', 'in', 'a', 'with', 'to', 'how', 'does', 'is', 'of'}
    search_terms = query_words - stop_words

    # 2. Score every community summary based on how many search terms it contains
    def score_summary(summary_text):
        if not isinstance(summary_text, str) or "Error" in summary_text:
            return 0
        summary_lower = summary_text.lower()
        # Count how many of the query keywords appear in this summary
        score = sum(1 for word in search_terms if word in summary_lower)
        return score

    # Create a new column with the relevance score
    df_comms['relevance_score'] = df_comms['summary'].apply(score_summary)

    # 3. SORT BY RELEVANCE FIRST, THEN BY SIZE!
    # This ensures we get the most relevant communities, but if there's a tie, we pick the bigger one.
    sorted_df = df_comms.sort_values(by=['relevance_score', 'size'], ascending=[False, False])

    valid_summaries = []
    for index, row in sorted_df.iterrows():
        summary = row.get('summary', '')
        score = row.get('relevance_score', 0)

        # Optional: You can skip summaries that have a score of 0 if you only want strict matches
        if isinstance(summary, str) and len(summary) > 20:
            valid_summaries.append(f"[Relevance: {score}] Theme {len(valid_summaries)+1}: {summary}")

        if len(valid_summaries) >= top_n:
            break

    print(f"Loaded {len(valid_summaries)} highly relevant community themes for context.")

    if not valid_summaries:
        return "No community summaries available.", []

    # 4. Final Output Generation
    context_str = "\n\n".join(valid_summaries)

    prompt_template = f"""
    You are a helpful medical assistant. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. Do not make up information.

    Context:
    {context_str}

    Question:
    {query}

    Answer:
    """

    print("Synthesizing final answer with Ollama (Llama)...")
    response = llm.invoke(prompt_template)

    # Clean up the dataframe so we don't permanently alter it
    df_comms.drop(columns=['relevance_score'], inplace=True, errors='ignore')

    return response.content, valid_summaries

# Loop

In [ ]:
import pandas as pd

# 1. The exact pathname to your Excel file
file_path = '/content/drive/MyDrive/genaiproj/evaluation.xlsx'

# 2. Tell Pandas exactly which tab (sheet) to open
sheet_name = 'graphrag-llama' # Change this to 'Professional' when you want to run the other tab!
df_eval = pd.read_excel(file_path, sheet_name=sheet_name)

generated_answers = []
retrieved_contexts = []
latency = []

print(f"Starting batch generation for {len(df_eval)} questions on the {sheet_name} sheet...")

# 3. The exact same loop as before
for index, row in df_eval.iterrows():
    question = row['Question']
    print(f"Processing Q{index + 1}: {question}")

    start = time.time()

    # Run the search
    answer, contexts = local_search(question, final_graph, llm)
    # answer, contexts = global_search(question, df_communities, llm)

    end_time = time.time()

    latency_time = end_time - start

    generated_answers.append(answer)
    retrieved_contexts.append(contexts)
    latency.append(latency_time)


# 4. Add the new columns
df_eval['Generated Answer'] = generated_answers
df_eval['Retrieved Contexts'] = retrieved_contexts
df_eval['Latency'] = latency

# 6. Save as a NEW Excel file so you don't overwrite your original empty one!
save_path = f'/content/drive/MyDrive/genaiproj/eval_graphrag_local_mistral_test1.xlsx'
df_eval.to_excel(save_path, index=False)
print(f"✅ Batch processing complete! Saved to {save_path}")

Starting batch generation for 97 questions on the graphrag-llama sheet...
Processing Q1: What are the primary tools used to assess glycemic status in a patient with diabetes?
Executing Local Search for: 'What are the primary tools used to assess glycemic status in a patient with diabetes?'
Detected query entities: ['DIABETES CARE', '2026 DIABETES CARE', 'GLYCEMIC GOALS', 'DIABETES', 'PEOPLE WITH DIABETES', 'AMERICAN DIABETES ASSOCIATION (ADA)', 'PROFESSIONAL DIABETES ORGANIZATION', 'AMERICAN DIABETES ASSOCIATION', 'ACTION IN DIABETES AND VASCULAR DISEASE: MODIFYING FACTORS FAVOR MORE STRINGENT GOAL', 'SHORT DIABETES DURATION', 'LONG DIABETES DURATION', 'PHARMACOLOGIC APPROACHES TO GLYCEMIC TREATMENT', 'DIABETES SELF-MANAGEMENT', 'INDIVIDUALS WITH DIABETES', 'PATIENT WITH DIABETES', 'TYPE 1 DIABETES', 'DIABETES SELF-MANAGEMENT EDUCATION AND SUPPORT PROGRAM', 'TRAINED DIABETES CARE AND EDUCATION SPECIALIST', 'DIABETES TREATMENT', 'DIABETES CARE AND EDUCATION SPECIALISTS', 'TYPE 2 DIABETE

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_local_subgraph(unique_edges, title="Local Search Subgraph"):
    """
    Takes a list of edge strings like 'DIABETES --[related_to]-- INSULIN'
    and draws a visual network graph.
    """
    # 1. Create an empty graph
    G = nx.DiGraph()

    # 2. Parse your specific string format and build the graph
    for edge_str in unique_edges:
        try:
            # Split "A --[rel]-- B" into its three parts
            part1, part2 = edge_str.split("]--")
            node_a, rel = part1.split("--[")

            node_a = node_a.strip()
            node_b = part2.strip()
            rel = rel.strip()

            # Add to the network
            G.add_edge(node_a, node_b, label=rel)
        except Exception as e:
            # Skip any strings that don't match the exact format perfectly
            continue

    # 3. If the graph is empty, stop here
    if G.number_of_nodes() == 0:
        print("No valid edges found to plot.")
        return

    # 4. Set up the canvas
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=0.8, iterations=50)
    nx.draw_networkx_nodes(G, pos, node_color='skyblue', node_size=3000, alpha=0.9, edgecolors='black')
    nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', font_family='sans-serif')
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15, alpha=0.6)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, font_color='red')

    plt.title(title, fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Define a test question
test_question = "What is the relationship between diabetes, insulin, and A1C?"

# 2. Run your search function (It should return the answer and your 1-item list context)
test_answer, test_context = local_search(test_question, df_communities, llm)

# 3. Extract the edges back out of that giant paragraph block
# (We just split it by line breaks to get the individual edges again)
raw_text_block = test_context[0]
edges_for_plotting = raw_text_block.split('\n')[1:] # Skip the "Retrieved Relationships:" header

# 4. Plot it!
print("Drawing the subgraph used to answer the question...")
plot_local_subgraph(edges_for_plotting, title=f"Graph for: '{test_question}'")

Executing Local Search for: 'What is the relationship between diabetes, insulin, and A1C?'


AttributeError: 'DataFrame' object has no attribute 'nodes'